# Adaptive Road Vision — BDD100K Dataset Verification

This notebook verifies the BDD100K dataset already placed inside the project. It checks the expected train/validation/test image structure, label files, JSON/JSONL annotation files, image-label matching, duplicate filenames, image readability, metadata/condition information when available, and the final dataset counts.

**Project path:** `C:\Users\ankit\Desktop\Adaptive_Road_Vision`

**Important:** This notebook is read-only with respect to the dataset. It does not create synthetic fog images, modify labels, or change the original train/val/test split.

## 1. Project Paths

The paths below match your current Windows project structure.

In [1]:
from pathlib import Path
import json
import csv
import hashlib
from collections import Counter, defaultdict
import sys

PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
BDD_ROOT = PROJECT_ROOT / "datasets" / "raw" / "BDD100K"
IMAGE_ROOT = BDD_ROOT / "images" / "100k"
LABEL_ROOT = BDD_ROOT / "labels"

SPLITS = ["train", "val", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LABEL_EXTENSIONS = {".json", ".jsonl"}

print("Project root:", PROJECT_ROOT)
print("BDD100K root :", BDD_ROOT)
print("Images root :", IMAGE_ROOT)
print("Labels root :", LABEL_ROOT)

Project root: C:\Users\ankit\Desktop\Adaptive_Road_Vision
BDD100K root : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K
Images root : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\images\100k
Labels root : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels


## 2. Verify Required Directories

In [2]:
required_paths = {
    "Project root": PROJECT_ROOT,
    "BDD100K root": BDD_ROOT,
    "Images root": IMAGE_ROOT,
    "Labels root": LABEL_ROOT,
}

print(f"{'STATUS':<10} {'NAME':<20} PATH")
print("-" * 100)
for name, path in required_paths.items():
    print(f"{'PASS' if path.exists() else 'FAIL':<10} {name:<20} {path}")

STATUS     NAME                 PATH
----------------------------------------------------------------------------------------------------
PASS       Project root         C:\Users\ankit\Desktop\Adaptive_Road_Vision
PASS       BDD100K root         C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K
PASS       Images root          C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\images\100k
PASS       Labels root          C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels


## 3. Count Images in Each Split

In [3]:
def image_files(folder):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])

image_files_by_split = {}
image_counts = {}

for split in SPLITS:
    files = image_files(IMAGE_ROOT / split)
    image_files_by_split[split] = files
    image_counts[split] = len(files)

print("BDD100K image counts")
for split in SPLITS:
    print(f"{split:5}: {image_counts[split]:,}")
print(f"TOTAL: {sum(image_counts.values()):,}")

expected = {"train": 70000, "val": 10000, "test": 20000}
print("\nExpected BDD100K counts:")
for split in SPLITS:
    actual = image_counts[split]
    target = expected[split]
    print(f"{split:5}: {'PASS' if actual == target else 'CHECK'} | actual={actual:,}, expected={target:,}")

BDD100K image counts
train: 70,000
val  : 10,000
test : 20,000
TOTAL: 100,000

Expected BDD100K counts:
train: PASS | actual=70,000, expected=70,000
val  : PASS | actual=10,000, expected=10,000
test : PASS | actual=20,000, expected=20,000


## 4. Inspect Image Filename Pattern

In [4]:
for split in SPLITS:
    files = image_files_by_split[split]
    print(f"\n{split.upper()} sample files:")
    for p in files[:10]:
        print(" ", p.name)


TRAIN sample files:
  0000f77c-6257be58.jpg
  0000f77c-62c2a288.jpg
  0000f77c-cb820c98.jpg
  0001542f-5ce3cf52.jpg
  0001542f-7c670be8.jpg
  0001542f-ec815219.jpg
  0004974f-05e1c285.jpg
  00054602-3bf57337.jpg
  00067cfb-5443fe39.jpg
  00067cfb-5adfaaa7.jpg

VAL sample files:
  b1c66a42-6f7d68ca.jpg
  b1c81faa-3df17267.jpg
  b1c81faa-c80764c5.jpg
  b1c9c847-3bda4659.jpg
  b1ca2e5d-84cf9134.jpg
  b1ca8418-84a133a0.jpg
  b1cac6a7-04e33135.jpg
  b1cd1e94-26dd524f.jpg
  b1cd1e94-549d0bfe.jpg
  b1ceb32e-3f481b43.jpg

TEST sample files:
  cabc30fc-e7726578.jpg
  cabc30fc-eb673c5a.jpg
  cabc30fc-fd79926f.jpg
  cabc9045-1b8282ba.jpg
  cabc9045-581f64de.jpg
  cabc9045-5a50690f.jpg
  cabc9045-b3349548.jpg
  cabc9045-c6dc9529.jpg
  cabc9045-cd422b81.jpg
  cabc9045-d91ecb66.jpg


## 5. Verify Labels / Annotation Files

This section supports both individual annotation files and JSON/JSONL files containing many annotations. It reports what is actually present rather than assuming a specific label format.

In [5]:
def label_files(folder):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in LABEL_EXTENSIONS])

label_files_by_split = {}
for split in SPLITS:
    files = label_files(LABEL_ROOT / split)
    label_files_by_split[split] = files
    extension_counts = Counter(p.suffix.lower() for p in files)
    print(f"{split.upper():5}: {len(files):,} annotation files | {dict(extension_counts)}")

all_label_files = label_files(LABEL_ROOT)
print(f"\nTotal JSON/JSONL annotation files: {len(all_label_files):,}")

TRAIN: 70,000 annotation files | {'.json': 70000}
VAL  : 10,000 annotation files | {'.json': 10000}
TEST : 20,000 annotation files | {'.json': 20000}

Total JSON/JSONL annotation files: 100,000


## 6. Inspect Label Directory Structure

In [6]:
for split in SPLITS:
    folder = LABEL_ROOT / split
    print(f"\n{split.upper()} label directory:", folder)
    if not folder.exists():
        print("  Directory does not exist.")
        continue
    for item in sorted(folder.iterdir())[:20]:
        kind = "DIR" if item.is_dir() else "FILE"
        print(f"  [{kind}] {item.name}")


TRAIN label directory: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels\train
  [FILE] 0000f77c-6257be58.json
  [FILE] 0000f77c-62c2a288.json
  [FILE] 0000f77c-cb820c98.json
  [FILE] 0001542f-5ce3cf52.json
  [FILE] 0001542f-7c670be8.json
  [FILE] 0001542f-ec815219.json
  [FILE] 0004974f-05e1c285.json
  [FILE] 00054602-3bf57337.json
  [FILE] 00067cfb-5443fe39.json
  [FILE] 00067cfb-5adfaaa7.json
  [FILE] 00067cfb-caba8a02.json
  [FILE] 00067cfb-e535423e.json
  [FILE] 00067cfb-f1b91e3c.json
  [FILE] 0008a165-c48f4b3e.json
  [FILE] 00091078-59817bb0.json
  [FILE] 00091078-7cff8ea6.json
  [FILE] 00091078-84635cf2.json
  [FILE] 00091078-875c1f73.json
  [FILE] 00091078-c1d32eea.json
  [FILE] 00091078-cedbfea7.json

VAL label directory: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels\val
  [FILE] b1c66a42-6f7d68ca.json
  [FILE] b1c81faa-3df17267.json
  [FILE] b1c81faa-c80764c5.json
  [FILE] b1c9c847-3bda4659.json
  [FILE] b1ca2e5d-84cf9134.json


## 7. Inspect JSON / JSONL Annotation Schema

The parser below reads only a small sample. It is designed to handle JSON objects, JSON arrays, and JSON Lines files.

In [7]:
def load_annotation_sample(path, max_records=3):
    try:
        text = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        text = path.read_text(encoding="utf-8-sig")

    stripped = text.lstrip()
    if not stripped:
        return []

    # JSON file: object or array
    if path.suffix.lower() == ".json":
        data = json.loads(text)
        if isinstance(data, list):
            return data[:max_records]
        return [data]

    # JSONL: one JSON object per line
    records = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue
        if len(records) >= max_records:
            break
    return records

sample_label_files = [p for split in SPLITS for p in label_files_by_split[split]][:5]

for path in sample_label_files:
    print("\nFILE:", path)
    try:
        records = load_annotation_sample(path)
        print("Records sampled:", len(records))
        for i, record in enumerate(records[:2]):
            if isinstance(record, dict):
                print(f"Record {i} keys:", list(record.keys())[:30])
            else:
                print(f"Record {i} type:", type(record).__name__)
    except Exception as e:
        print("Could not parse sample:", repr(e))


FILE: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels\train\0000f77c-6257be58.json
Records sampled: 1
Record 0 keys: ['name', 'frames', 'attributes']

FILE: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels\train\0000f77c-62c2a288.json
Records sampled: 1
Record 0 keys: ['name', 'frames', 'attributes']

FILE: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels\train\0000f77c-cb820c98.json
Records sampled: 1
Record 0 keys: ['name', 'frames', 'attributes']

FILE: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels\train\0001542f-5ce3cf52.json
Records sampled: 1
Record 0 keys: ['name', 'frames', 'attributes']

FILE: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels\train\0001542f-7c670be8.json
Records sampled: 1
Record 0 keys: ['name', 'frames', 'attributes']


## 8. Search for BDD100K Metadata / Attribute Information

Condition analysis later needs Day/Night and weather information. This cell searches the BDD100K root for likely metadata or attribute files without assuming their exact filename.

In [8]:
metadata_keywords = ("attr", "attribute", "metadata", "label", "train", "val", "test")
metadata_candidates = []

for p in BDD_ROOT.rglob('*'):
    if p.is_file() and p.suffix.lower() in {".json", ".jsonl", ".csv", ".txt"}:
        name = p.name.lower()
        if any(k in name for k in metadata_keywords):
            metadata_candidates.append(p)

print(f"Potential metadata/annotation files found: {len(metadata_candidates):,}")
for p in metadata_candidates[:50]:
    print(" ", p.relative_to(BDD_ROOT))

Potential metadata/annotation files found: 0


## 9. Extract Condition Information from Annotation Samples

In [9]:
CONDITION_KEYS = {
    "weather", "timeofday", "time_of_day", "timeofday", "scene_weather",
    "scene", "attributes", "weather_condition", "lighting", "condition"
}

def find_values(obj, key_path=""):
    found = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            normalized = str(key).lower().replace("-", "_").replace(" ", "_")
            current_path = f"{key_path}.{key}" if key_path else str(key)
            if normalized in CONDITION_KEYS:
                found.append((current_path, value))
            found.extend(find_values(value, current_path))
    elif isinstance(obj, list):
        for i, value in enumerate(obj[:20]):
            found.extend(find_values(value, f"{key_path}[{i}]"))
    return found

condition_observations = []
for path in sample_label_files:
    try:
        records = load_annotation_sample(path, max_records=5)
        for record in records:
            condition_observations.extend(find_values(record))
    except Exception:
        pass

if condition_observations:
    for key, value in condition_observations[:50]:
        print(key, "=", value)
else:
    print("No condition fields were found in the sampled annotation files.")
    print("This is not automatically an error; the BDD100K attributes may be stored in a separate metadata file.")

frames[0].objects[0].attributes = {'occluded': False, 'truncated': False, 'trafficLightColor': 'green'}
frames[0].objects[1].attributes = {'occluded': False, 'truncated': False, 'trafficLightColor': 'green'}
frames[0].objects[2].attributes = {'occluded': False, 'truncated': False, 'trafficLightColor': 'none'}
frames[0].objects[3].attributes = {'occluded': False, 'truncated': True, 'trafficLightColor': 'none'}
frames[0].objects[4].attributes = {'occluded': False, 'truncated': False, 'trafficLightColor': 'none'}
frames[0].objects[5].attributes = {'occluded': False, 'truncated': False, 'trafficLightColor': 'none'}
frames[0].objects[6].attributes = {'occluded': False, 'truncated': True, 'trafficLightColor': 'none'}
frames[0].objects[7].attributes = {}
frames[0].objects[8].attributes = {'direction': 'parallel', 'style': 'solid'}
frames[0].objects[9].attributes = {'direction': 'parallel', 'style': 'solid'}
frames[0].objects[10].attributes = {'direction': 'parallel', 'style': 'solid'}
attribu

## 10. Check Duplicate Image Names Across Splits

Duplicate names across train/validation/test can be a warning sign. The check uses filenames only; later, sequence-aware verification should be performed before training.

In [10]:
name_to_splits = defaultdict(list)
for split, files in image_files_by_split.items():
    for p in files:
        name_to_splits[p.name].append(split)

cross_split_duplicates = {
    name: splits for name, splits in name_to_splits.items() if len(set(splits)) > 1
}

print("Cross-split duplicate filenames:", len(cross_split_duplicates))
for name, splits in list(cross_split_duplicates.items())[:20]:
    print(" ", name, "->", splits)

if not cross_split_duplicates:
    print("PASS: No duplicate filenames were found across train/val/test.")

Cross-split duplicate filenames: 0
PASS: No duplicate filenames were found across train/val/test.


## 11. Verify Image Readability

In [11]:
def verify_images(files, limit=None):
    try:
        from PIL import Image
    except ImportError:
        print("Pillow is not installed; image readability test skipped.")
        return []

    check_files = files if limit is None else files[:limit]
    bad = []
    for i, path in enumerate(check_files, 1):
        try:
            with Image.open(path) as img:
                img.verify()
        except Exception as e:
            bad.append((path, str(e)))
    return bad

print("Checking up to 100 images per split...")
for split in SPLITS:
    bad = verify_images(image_files_by_split[split], limit=100)
    print(f"{split:5}: {'PASS' if not bad else 'CHECK'} | bad images in sample = {len(bad)}")
    for path, error in bad[:5]:
        print(" ", path, "->", error)

Checking up to 100 images per split...
train: PASS | bad images in sample = 0
val  : PASS | bad images in sample = 0
test : PASS | bad images in sample = 0


## 12. Image Resolution Sample

In [12]:
try:
    from PIL import Image
    resolution_counter = Counter()
    sampled = []
    for split in SPLITS:
        for path in image_files_by_split[split][:100]:
            try:
                with Image.open(path) as img:
                    resolution_counter[img.size] += 1
                    sampled.append((split, path.name, img.size))
            except Exception:
                pass

    print("Most common sampled resolutions:")
    for resolution, count in resolution_counter.most_common(10):
        print(f"  {resolution[0]} x {resolution[1]} : {count}")
except ImportError:
    print("Pillow is not installed; resolution check skipped.")

Most common sampled resolutions:
  1280 x 720 : 300


## 13. Check Empty / Unexpected Files

This identifies files with zero bytes and files that are not recognized as normal image or annotation files inside the main dataset directories.

In [13]:
def zero_byte_files(root):
    return [p for p in root.rglob('*') if p.is_file() and p.stat().st_size == 0]

zero_images = zero_byte_files(IMAGE_ROOT) if IMAGE_ROOT.exists() else []
zero_labels = zero_byte_files(LABEL_ROOT) if LABEL_ROOT.exists() else []

print("Zero-byte image files:", len(zero_images))
print("Zero-byte label files:", len(zero_labels))

for p in zero_images[:10]:
    print(" IMAGE:", p)
for p in zero_labels[:10]:
    print(" LABEL:", p)

Zero-byte image files: 0
Zero-byte label files: 0


## 14. Dataset Summary

The values below summarize the original dataset before any CycleGAN augmentation. Do not add synthetic fog images here.

In [14]:
total_images = sum(image_counts.values())
total_labels = sum(len(v) for v in label_files_by_split.values())

print("=" * 70)
print("BDD100K DATASET SUMMARY")
print("=" * 70)
print(f"Train images      : {image_counts['train']:,}")
print(f"Validation images : {image_counts['val']:,}")
print(f"Test images       : {image_counts['test']:,}")
print(f"Total images      : {total_images:,}")
print(f"Annotation files  : {total_labels:,}")
print("Synthetic fog     : NOT CREATED IN THIS NOTEBOOK")

if total_images == 100000:
    print("\nPASS: Total image count is 100,000.")
else:
    print("\nCHECK: Total image count is different from the expected 100,000.")

BDD100K DATASET SUMMARY
Train images      : 70,000
Validation images : 10,000
Test images       : 20,000
Total images      : 100,000
Annotation files  : 100,000
Synthetic fog     : NOT CREATED IN THIS NOTEBOOK

PASS: Total image count is 100,000.


## 15. Verification Checklist

Before moving to `03_jsonl_annotation_processing.ipynb`, confirm:

- [ ] Train contains approximately/exactly 70,000 images.
- [ ] Validation contains approximately/exactly 10,000 images.
- [ ] Test contains approximately/exactly 20,000 images.
- [ ] Original test data remains untouched.
- [ ] Annotation files are present and readable.
- [ ] No cross-split duplicate filenames were detected.
- [ ] Image readability sample passes.
- [ ] BDD100K condition metadata is located or its absence is clearly understood.
- [ ] No CycleGAN data has been added yet.

**Next notebook:** `03_jsonl_annotation_processing.ipynb`